# Workshop 4 – Science Theme: Building a Simple RAG Pipeline

In this notebook, we connect retrieval and generation into one complete pipeline.

We will:
- load the saved FAISS database
- retrieve relevant scientific context for a query
- build a prompt from that context
- generate an answer with Qwen

This gives us a simple example of a Retrieval-Augmented Generation (RAG) system.

## Step 1: Install and Import Libraries

We will use the same main tools as before:

- **LangChain** to load the saved FAISS database
- **Transformers** to run the Qwen model
- **Sentence Transformers** for embeddings
- **FAISS** for similarity search

In [1]:
!pip install -q langchain langchain-community transformers sentence-transformers faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.


In [2]:
#Importing the libraries
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline

## Step 2: Load the Saved FAISS Database

We already created the vector database in Workshop 2.

So here, we simply load the saved FAISS index and reuse it for retrieval.

In [3]:
embedding_model = HuggingFaceEmbeddings()

db = FAISS.load_local(
    "faiss_Sci",
    embedding_model,
    allow_dangerous_deserialization=True
)

print("FAISS index loaded successfully.")

/tmp/ipykernel_3388/2393039932.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings()
/tmp/ipykernel_3388/2393039932.py:1: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding_model = HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/s

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index loaded successfully.


## Step 3: Define a Retrieval Function

The first part of a RAG pipeline is retrieval.

We define a small function that takes a user query and returns the most similar chunks from the FAISS database.

In [5]:
def retrieve_docs(query, k=3):#default value k=3
    docs = db.similarity_search(query, k=k)
    return docs

## Step 4: Test Retrieval

Before building the full pipeline, we first check that retrieval is working correctly.

In [6]:
query = "neural networks for image classification"

results = retrieve_docs(query, k=5)

for i, doc in enumerate(results, 1):
    print(f"\nResult {i}:\n")
    print(doc.page_content[:700])


Result 1:

Support Vector Machines (SVMs). In this study it was observed that the ANNs
perform better than the SVMs. This performance is measured against the
generalisation ability of the two.

Result 2:

Comparing Robustness of Pairwise and Multiclass Neural-Network Systems
  for Face Recognition   Noise, corruptions and variations in face images can seriously hurt the
performance of face recognition systems. To make such systems robust,
multiclass neuralnetwork classifiers capable of learning from noisy data have
been suggested. However on large face data sets such systems cannot provide the
robustness at a high level. In this paper we explore a pairwise neural-network
system as an alternative approach to improving the robustness of face
recognition. In our experiments this approach is shown to outperform the
multiclass neural-network system in terms of the predictive accuracy on the
face i

Result 3:

The Parameter-Less Self-Organizing Map algorithm   The Parameter-Less Self-Organi

## Step 5: Format Retrieved Chunks as Context

The retriever returns a list of document chunks.

Before passing them to the language model, we combine them into one text block.
This will form the context for the final prompt.

In [7]:
def format_docs(docs):
    context = "\n\n".join([doc.page_content for doc in docs])
    return context

In [8]:
context = format_docs(results)

print(context[:1000])#Preview the context after merging the retrieved documents

Support Vector Machines (SVMs). In this study it was observed that the ANNs
perform better than the SVMs. This performance is measured against the
generalisation ability of the two.

Comparing Robustness of Pairwise and Multiclass Neural-Network Systems
  for Face Recognition   Noise, corruptions and variations in face images can seriously hurt the
performance of face recognition systems. To make such systems robust,
multiclass neuralnetwork classifiers capable of learning from noisy data have
been suggested. However on large face data sets such systems cannot provide the
robustness at a high level. In this paper we explore a pairwise neural-network
system as an alternative approach to improving the robustness of face
recognition. In our experiments this approach is shown to outperform the
multiclass neural-network system in terms of the predictive accuracy on the
face images corrupted by noise.

The Parameter-Less Self-Organizing Map algorithm   The Parameter-Less Self-Organizing Map 

## Step 6: Load the Qwen Model

We now load the same model used earlier.

This keeps the focus on the new idea in this workshop:
combining retrieval and generation in one pipeline.

In [4]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

generator = pipeline(
    "text-generation",
    model=model_name,
    tokenizer=model_name
)

print("Model loaded successfully.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded successfully.


## Step 7: Build a Prompt Template

A prompt template helps us control how the model responds.

Here, we ask the model to:
- use the retrieved context
- avoid guessing
- return a clear answer

In [30]:
def build_prompt(context, question, allow_model_knowledge=False):

    if allow_model_knowledge==False:
        instruction = """
First, check if the context contains the answer.
If yes, answer using the context.
If not, return ONLY: I don't know.
Do not attempt to answer before checking.
"""
    else:
        instruction = """
Use the context below as your main source of information.
You may use your own knowledge to explain and clarify the answer.

Do not invent references.
"""

    prompt = f"""
You are a scientific assistant.

{instruction}

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [32]:
question = "What approaches are used for image classification in this context?"

prompt = build_prompt(context, question, allow_model_knowledge=False)

print(prompt)


You are a scientific assistant.


Use the context below as your main source of information.
You may use your own knowledge to explain and clarify the answer.

Do not invent references.


Context:
Support Vector Machines (SVMs). In this study it was observed that the ANNs
perform better than the SVMs. This performance is measured against the
generalisation ability of the two.

Comparing Robustness of Pairwise and Multiclass Neural-Network Systems
  for Face Recognition   Noise, corruptions and variations in face images can seriously hurt the
performance of face recognition systems. To make such systems robust,
multiclass neuralnetwork classifiers capable of learning from noisy data have
been suggested. However on large face data sets such systems cannot provide the
robustness at a high level. In this paper we explore a pairwise neural-network
system as an alternative approach to improving the robustness of face
recognition. In our experiments this approach is shown to outperform the
mu

In [22]:
def generate_answer(prompt, max_new_tokens=200):
    response = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    full_text = response[0]["generated_text"]

    # Remove the prompt part
    answer = full_text[len(prompt):].strip()

    return answer

In [25]:
answer = generate_answer(prompt)

print("Answer:\n",answer)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:
 I don't know.


## Step 8: Build the Full RAG Pipeline

We now combine all steps into one function:

1. retrieve relevant chunks
2. format them into context
3. build the prompt
4. generate the answer

This is the core idea of a simple RAG system.

In [33]:
def rag_pipeline(question, k=3, allow_model_knowledge=False):
    docs = retrieve_docs(question, k=k)
    context = format_docs(docs)

    prompt = build_prompt(
        context,
        question,
        allow_model_knowledge=allow_model_knowledge
    )

    answer = generate_answer(prompt)
    return answer

In [35]:
query = "What approaches are used for image classification?"

response = rag_pipeline(query, k=3, allow_model_knowledge=False)

print(response)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The paper does not mention any specific approaches for image classification. It focuses on the limitations of existing classification models and proposes an adaptive strategy for incorporating new domain knowledge using incremental learning techniques. However, without more details about the methods described, I cannot provide a comprehensive answer regarding what approaches are used for image classification. If you have additional context or information related to image classification, please share it so I can assist you further. Otherwise, I will focus on the approach mentioned in the given context: utilizing fuzzy ARTMAP for incremental learning and an evolutionary strategy for ensemble system selection and combination. Let me know if there's anything else I can help with regarding image classification.
